In [ ]:
# Cell 0 — Imports & Configuration
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# --- Paths ---
DATA_ROOT       = Path("../../../data/00-newspaper_data")
CHECKPOINT_PATH = DATA_ROOT / "processed/daily_summaries_checkpoint.parquet"
FINAL_OUTPUT    = DATA_ROOT / "processed/daily_summaries.parquet"
FAILED_CSV      = DATA_ROOT / "processed/daily_summaries_failed.csv"

# --- Config ---
DATE_RANGE     = ("2018-01-01", "2024-09-30")  # matches periodistas dataset
MODEL          = "gpt-4o-mini"
RETRY_ATTEMPTS = 3
RETRY_DELAY    = 5      # seconds; doubles on each attempt
SAVE_INTERVAL  = 50     # save checkpoint every N days processed

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# --- Extensible sources config ---
# To add a new newspaper: add an entry here. No other changes needed.
# 'summary_col': column to use as text (falls back to title_col if null/empty).
#                Set to None to always use title_col.
# 'tz_aware':    True if the parquet stores UTC-aware datetimes (24H files).
# 'label':       Editorial description shown to GPT in the prompt.
SOURCES = {
    "jornada": {
        "path":           DATA_ROOT / "crawler/jornada/articles_final.parquet",
        "title_col":      "title",
        "summary_col":    "summary",   # use summary when available
        "topic_col":      "topic",
        "tz_aware":       False,
        "label":          "La Jornada (línea editorial afín al gobierno)",
        "topic_allowlist": {
            "politica", "opinion", "economia", "estados", "capital", "sociedad",
        },
    },
    "cdmx_24h": {
        "path":           DATA_ROOT / "crawler/horas24/articles_cdmx_final.parquet",
        "title_col":      "title",
        "summary_col":    None,         # titles only for 24H
        "topic_col":      "topic",
        "tz_aware":       True,
        "label":          "24 Horas (línea editorial opositora)",
        "topic_allowlist": {
            # --- Política ---
            "Política", "AMLO", "EPN", "Sheinbaum", "Gobernadores", "Alcaldes",
            "Alcaldes y Gobernadores", "Desde las Cámaras Legislativas", "Congreso",
            "Elecciones 2018", "Elecciones 2021", "Elecciones 2022",
            "Elecciones EU", "Elecciones EUA", "Elecciones Edomex", "Elecciones MX",
            "Elección 2019", "Donald Trump", "AMLO Rinde Protesta", "Consulta NAIM",
            "revocación de mandato", "Reforma Eléctrica", "Reforma electoral",
            "Cuarto informe de AMLO", "Marcha por el INE",
            "La Divisa del Poder", "Teléfono Rojo", "Hechos y Susurros",
            "Nomenklatura del Poder", "Actos de Poder", "Tablero Político",
            "Itinerario Politico", "Votos y Billetes", "Reporte Lobby",
            "Indebidos Procesos", "Emilio Lozoya", "Javier Duarte", "Roberto Borge",
            "Alfredo del Mazo", "Fideicomisos", "Guacamaya Leaks",
            "Genaro García Luna", "Salvador Cienfuegos",
            "Captura de Ovidio Guzmán", "Captura de Caro Quintero",
            "Opinión", "Columnas", "México", "LOS OTROS DATOS",
            "EXPEDIENTE", "SIN DISTORSIONES", "Obituario - Política",
            # --- Seguridad ---
            "Justicia", "Estado-Justicia", "Ciudad-Justicia", "Seguridad",
            "Agenda de Seguridad y Defensa", "Guardia Nacional",
            "Feminicidio", "Feminicidios", "Derechos Humanos", "Ayotzinapa",
            "El Chapo", "Cerocahui", "sabinas", "Ariadna Fernanda",
            "Debanhi Escobar", "Desaparecidos", "Caso Zona Rosa", "Balacera AICM",
            "Migración", "Caravana Migrante", "Caravana Migrante pf",
            "Sin Fronteras", "Migrantes",
            # --- Economía ---
            "Economía", "Economia", "econo", "Indicadores economicos",
            "Negocios", "Negocios - columnas", "Finanzas24 y Negocios",
            "T-MEC", "USMCA", "Pemex", "CFE", "Petróleo",
            "Desabasto de combustibles", "Split Financiero",
            "Coronavirus Economía", "Apuntes Macro", "AIFA", "Aeropuerto",
            "Tren Maya", "Empresarios del Campo", "Presupuesto 2020",
            "Desde la Banca", "Desde el piso de remates",
            # --- Estados ---
            "Estados", "CDMX", "Quintana Roo", "San Luis Potosí",
            "Aguascalientes", "Guerrero", "Puebla", "Edomex", "Sinaloa",
            "Sonora", "Querétaro", "Baja California Sur", "Yucatán",
            "Hidalgo", "Tlaxcala", "Durango", "Naucalpan", "Oaxaca",
            "Morelos", "Nezahualcóyotl", "Campeche", "Jalisco", "Coahuila",
            "Tamaulipas", "Nuevo León", "Guanajuato", "Chiapas", "Huixquilucan",
        },
    },
}

print(f"Sources configured: {list(SOURCES.keys())}")
print(f"Date range: {DATE_RANGE[0]} → {DATE_RANGE[1]}")

In [ ]:
# Cell 1 — Load, Filter, and Build Daily Text Collections

def get_text_per_article(row, cfg):
    """Return summary if available, otherwise title."""
    if cfg["summary_col"]:
        s = str(row.get(cfg["summary_col"]) or "").strip()
        if s:
            return s
    return str(row.get(cfg["title_col"]) or "").strip()


def load_source(source_name, cfg):
    """Load one source parquet, apply date + topic filters, return (date, label, text) rows."""
    df = pd.read_parquet(cfg["path"])
    df = df.drop_duplicates(subset="url")

    if cfg["tz_aware"]:
        df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
    else:
        df["date"] = pd.to_datetime(df["date"])

    df = df[(df["date"] >= DATE_RANGE[0]) & (df["date"] <= DATE_RANGE[1])]
    df["date_only"] = df["date"].dt.date

    topic_col = cfg["topic_col"]
    df[topic_col] = df[topic_col].fillna("").astype(str)
    df = df[df[topic_col].isin(cfg["topic_allowlist"])].copy()

    df["_text"]  = [get_text_per_article(row, cfg) for row in df.to_dict("records")]
    df["_label"] = cfg["label"]
    df = df[df["_text"].str.len() > 0]  # drop empty texts

    print(f"  {source_name}: {len(df):,} articles after filter")
    return df[["date_only", "_label", "_text"]]


print("Loading sources...")
frames = [load_source(name, cfg) for name, cfg in SOURCES.items()]
all_articles = pd.concat(frames, ignore_index=True)

# Group by calendar day → list of (label, text) tuples
daily = (
    all_articles
    .assign(_entry=lambda d: list(zip(d["_label"], d["_text"])))
    .groupby("date_only")["_entry"]
    .apply(list)
    .reset_index()
    .rename(columns={"date_only": "date", "_entry": "entries"})
)

print(f"\nDays with articles: {len(daily):,}")
print(f"Total article texts: {all_articles.shape[0]:,}")
print(f"Avg articles/day: {all_articles.shape[0] / len(daily):.1f}")
daily.head(3)

In [ ]:
# Cell 2 — Prompt Design

SYSTEM_PROMPT = (
    "Eres un asistente especializado en el análisis de noticias mexicanas.\n"
    "Se te proporcionará una lista numerada de artículos periodísticos publicados "
    "en un mismo día en México, provenientes de dos fuentes con distintas líneas editoriales:\n"
    "- La Jornada: periódico de izquierda con línea editorial afín al gobierno de AMLO.\n"
    "- 24 Horas: periódico con línea editorial opositora al gobierno.\n\n"
    "Tu tarea: redactar UN párrafo en español (máximo 150 palabras) que describa "
    "los principales eventos, temas y noticias del día cubiertos en esos artículos.\n"
    "Enfócate en política, economía, seguridad y asuntos de gobierno.\n"
    "Sé específico: menciona actores, instituciones y eventos concretos cuando aparezcan.\n"
    "Puedes señalar si un mismo evento recibe cobertura distinta según la fuente, "
    "pero no es obligatorio.\n"
    "No hagas listas. Solo el párrafo, sin introducción ni conclusión."
)


def build_user_message(entries):
    """
    entries: list of (label, text) tuples, one per article.
    Groups them by label so GPT sees each source as a block.
    """
    from collections import defaultdict
    by_source = defaultdict(list)
    for label, text in entries:
        by_source[label].append(text)

    blocks = []
    for label, texts in by_source.items():
        numbered = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(texts) if t)
        blocks.append(f"[{label}]\n{numbered}")

    return "Artículos del día:\n\n" + "\n\n".join(blocks)


# Quick sanity check
sample_day = daily.iloc[0]
print(f"Sample date: {sample_day['date']}")
print(f"Articles in sample: {len(sample_day['entries'])}")
print()
print("--- SYSTEM PROMPT ---")
print(SYSTEM_PROMPT)
print()
print("--- USER MESSAGE (first 3 per source) ---")
# Show a trimmed version for readability
from collections import defaultdict
preview = defaultdict(list)
for label, text in sample_day["entries"]:
    if len(preview[label]) < 3:
        preview[label].append(text)
trimmed = [(lbl, txt) for lbl, txts in preview.items() for txt in txts]
print(build_user_message(trimmed))

In [ ]:
# Cell 3 — API Call with Retry

def call_api(entries):
    """
    Send one day's article entries (list of (label, text) tuples) to GPT.
    Returns the summary paragraph, or None if all retries fail.
    """
    user_msg = build_user_message(entries)
    for attempt in range(RETRY_ATTEMPTS):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0,
                max_tokens=300,
                timeout=30,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"  [error] attempt {attempt + 1}: {e}")
            time.sleep(wait)
    return None


# Test with the sample day
print(f"Testing API call for {sample_day['date']} ({len(sample_day['entries'])} articles)...")
test_summary = call_api(sample_day["entries"])
print()
print("--- GPT OUTPUT ---")
print(test_summary)

In [ ]:
# Cell 4 — Checkpoint Load

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        df   = pd.read_parquet(CHECKPOINT_PATH)
        done = set(df["date"].astype(str).tolist())
        print(f"Checkpoint loaded: {len(done):,} dates already processed")
        return df.to_dict("records"), done
    print("No checkpoint found — starting fresh")
    return [], set()


all_results, processed_dates = load_checkpoint()

In [ ]:
# Cell 5 — Main Processing Loop

failed_dates = []
remaining    = daily[~daily["date"].astype(str).isin(processed_dates)]
print(f"To process: {len(remaining):,} | Already done: {len(processed_dates):,}")

for i, row in enumerate(tqdm(remaining.itertuples(index=False), total=len(remaining), desc="Daily summaries")):
    summary = call_api(row.entries)

    if summary is None:
        failed_dates.append(str(row.date))
    else:
        all_results.append({
            "date":          row.date,
            "daily_summary": summary,
            "n_articles":    len(row.entries),
        })
        processed_dates.add(str(row.date))

    if (i + 1) % SAVE_INTERVAL == 0:
        pd.DataFrame(all_results).to_parquet(CHECKPOINT_PATH, index=False)
        tqdm.write(f"  Checkpoint saved ({i + 1} days done, {len(failed_dates)} failed)")

# Final checkpoint save
pd.DataFrame(all_results).to_parquet(CHECKPOINT_PATH, index=False)
print(f"\nDone. {len(all_results):,} days processed | {len(failed_dates)} failures")

In [ ]:
# Cell 6 — Build Final Output & Validate

df = pd.DataFrame(all_results)
df["date"]       = pd.to_datetime(df["date"])
df["n_articles"] = df["n_articles"].astype(int)
df = df.sort_values("date").reset_index(drop=True)

(DATA_ROOT / "processed").mkdir(parents=True, exist_ok=True)
df.to_parquet(FINAL_OUTPUT, index=False)

print("=== Summary ===")
print(f"Shape:             {df.shape}")
print(f"Date range:        {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Avg articles/day:  {df['n_articles'].mean():.1f}")
print(f"Min articles/day:  {df['n_articles'].min()}")
print(f"Max articles/day:  {df['n_articles'].max()}")
print(f"\nSaved → {FINAL_OUTPUT}")

if failed_dates:
    pd.Series(failed_dates, name="failed_date").to_csv(FAILED_CSV, index=False)
    print(f"Failed dates ({len(failed_dates)}) → {FAILED_CSV}")

print("\n--- Sample output ---")
df[["date", "n_articles", "daily_summary"]].head(5)